# AtaRehber: evaluation on Kaggle

Runs `python -m evals.run_eval` on a Kaggle GPU and saves the report under `/kaggle/working/results`.

**Before running** (right-hand panel, *Session options*):
* **Accelerator:** GPU T4 x2 (or P100)
* **Internet:** On (needs a phone-verified Kaggle account)

Reports are kept only if you download them or run the notebook with **Save Version → Save & Run All**; files of an
interactive session are deleted when it ends.

In [ ]:
# Settings
REPO = "https://github.com/SirAlper/AtaRehber.git"
BRANCH = "main"
MODEL = "qwen2.5:7b"          # e.g. "qwen2.5:14b" (about 9 GB, fits a 16 GB T4)
DATASET = "university"        # "university" (127 questions, laws and regulations) or "demo" (51 questions)
STAGES = "all"                # "retrieval", "retrieval,routing", "all", "agents", "grader"
IDS = ""                      # e.g. "u-rek-01,u-rek-02" for a quick trial; empty: every question

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print("WARNING: the project is tested on Python 3.12+; this image is older and may fail.")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and Python packages

PyTorch is not installed: the image's own build (with CUDA) is used.

In [ ]:
%cd /kaggle/working
!rm -rf AtaRehber && git clone --depth 1 --branch {BRANCH} {REPO} AtaRehber
%cd /kaggle/working/AtaRehber
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Ollama and the language model

In [ ]:
import subprocess, time, urllib.request

!apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /kaggle/working/ollama.log")
!ollama pull {MODEL} 2>&1 | tail -1

## 3. Embedding and reranker models (about 3.3 GB)

In [ ]:
!python download_model.py 2>&1 | tail -5

## 4. Evaluation

Each question prints a line when it starts and one when it is done, with the progress and the time left.

In [ ]:
import os

os.environ.update({"OLLAMA_MODEL": MODEL, "RAG_DEVICE": "cuda"})
args = f"--stages {STAGES} --out /kaggle/working/results"
if DATASET == "university":
    # The threshold measured for article-aware chunks (docs/evaluation.md)
    os.environ["RAG_MIN_RERANKER_SCORE"] = "0.05"
    args += " --dataset evals/dataset_university.jsonl --corpus evals/corpus_university --no-database"
if IDS:
    args += f" --ids {IDS}"
print("python -m evals.run_eval", args)
!python -m evals.run_eval {args} 2>&1 | grep -v -E "^\[20|Batches|Loading weights"

## 5. Summary of the newest report

In [ ]:
import glob, json

reports = sorted(glob.glob("/kaggle/working/results/report_*.json"))
if not reports:
    raise SystemExit("No report was written; see the output above.")
report = json.load(open(reports[-1], encoding="utf-8"))
print(reports[-1])
print("commit:", report["git_commit"], "| model:", report["config"]["llm_model"], "| cases:", report["case_count"])
for stage, summary in report["summary"].items():
    print(f"\n== {stage} ==")
    for key, value in summary.items():
        if not isinstance(value, (dict, list)):
            print(f"  {key:<34} {value}")
!ollama ps